In [0]:
BASE_DIR = '/Volumes/sales/landing/sales/OrderPayments/'
CHECKPOINT_DIR = '/Volumes/sales/landing/sales/CheckPoint_Directory/'
CHECKPOINT_LOCATION = f'{CHECKPOINT_DIR}/OrderPayments'
CATALOG = 'Sales'
SCHEMA = 'Bronze'
TARGET_TABLE = 'OrderPayments'
print(f'checkpoint_location : {CHECKPOINT_LOCATION}')
print(f'BASE_LOCATION = {BASE_DIR}')

In [0]:
OP_schema = '''order_id STRING, 
               payment_sequential INT, 
               payment_type STRING, 
               payment_installments INT, 
               payment_value DOUBLE'''

In [0]:
OP_df = spark.read.format('csv')\
                  .option('header', 'true')\
                  .schema(OP_schema)\
                  .load(f'{BASE_DIR}')
display(OP_df.limit(3))
OP_df.printSchema()

%md
#### LOAD THE `OrderPayments` DATA INTO BRONZE LAYER
> Order_Sales . Bronze . printSchema

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
OP_df = (
            spark.readStream.format('cloudFiles')
                 .option('cloudFiles.format', 'csv')
                 .option('cloudFiles.schemaEvolutionMode', 'rescue')
                 .option('header', 'true')
                 .schema(OP_schema)
                 .load(f'{BASE_DIR}')
        )
OP_trans_df = OP_df.select(
    "order_id", 
    "payment_sequential", 
    "payment_type", 
    "payment_installments", 
    "payment_value",
    "_metadata.file_name",
    "_metadata.file_path",).withColumn('load_timestamp', current_timestamp())

#OP_trans_df.printSchema()
#OP_trans_df.limit(3).display(checkpointLocation=f'{CHECKPOINT_DIR}/OP_display')

In [0]:
qry = (
    OP_trans_df.writeStream.format('delta')
    .outputMode('append')
    .option('checkpointLocation', CHECKPOINT_LOCATION)
    .trigger(availableNow=True)
    .toTable(f'{CATALOG}.{SCHEMA}.{TARGET_TABLE}')
)
qry.awaitTermination()